# Préparation, modélisation et évaluation — Membre 2 : Asma Abdellatif
## Sujet : Prédire si le séjour sera **court ou long** (classification binaire sur `time_in_hospital`)

**Projet :** DiaCare Analytics — Équipe 3 (4CCE19, ESPRIT)

Ce notebook reprend `df_clean.csv` (sortie du nettoyage du Membre 1 : un séjour par patient,
patients décédés retirés, valeurs manquantes traitées), réalise la **préparation des données**,
puis entraîne et évalue deux modèles de **classification** : **XGBoost** et **AdaBoost**.

> Changement par rapport à la version précédente : la régression de la durée (en jours) est
> remplacée par une classification à deux classes, « courte durée » / « longue durée », afin
> de mesurer l'**accuracy**.

## 0. Objectifs (rappel)

**Objectif métier :** prévoir l'occupation des lits pour planifier le personnel et les
admissions, en repérant les patients qui auront un **séjour long**.
Critère de réussite : une accuracy nettement supérieure à la référence (toujours prédire la
classe majoritaire), avec un bon **rappel** de la classe « longue durée » (un séjour long non
repéré coûte plus cher en lits mal planifiés qu'une fausse alerte).

**Objectif data science :**
- Type de problème : **classification binaire supervisée**
- Variable cible : `long_stay` = 1 si `time_in_hospital` > `SEUIL_JOURS` (**longue durée**), 0 sinon
  (**courte durée**). Le seuil par défaut est de **3 jours**, la médiane : les deux classes sont
  alors à peu près équilibrées (environ 50 / 50, à vérifier en section 1.3), donc l'accuracy est
  interprétable.
- À exclure des variables explicatives : `readmitted` (information future) et `time_in_hospital`
  (la cible en est dérivée : la garder serait de la fuite d'information).
  `num_lab_procedures` et `num_medications` ne sont connus qu'en fin de séjour → gardés
  uniquement dans la version *« après séjour »*
- Métrique principale : **accuracy**. Métriques complémentaires : précision, rappel, F1, AUC,
  matrice de confusion
- Modèles : **XGBoost** et **AdaBoost**, comparés à une référence (classe majoritaire)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier   # préinstallé sur Colab ; sinon : !pip install xgboost
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

# Seuil entre séjour court et long : long = strictement plus de SEUIL_JOURS jours
SEUIL_JOURS = 3

## 1. Nettoyage — point de départ

### 1.1 Chargement de `df_clean` (sortie du Membre 1)

In [ ]:
# Sur Colab : uploader df_clean.csv dans l'environnement, ou monter Google Drive.
# from google.colab import files
# files.upload()

df = pd.read_csv("df_clean.csv")
print("Dimensions :", df.shape)
df.head()


In [ ]:
df.info()


### 1.2 Visualisation — distribution de la variable cible `time_in_hospital`

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["time_in_hospital"], bins=14, kde=True, color="steelblue")
plt.title("Distribution de la durée de séjour (time_in_hospital)")
plt.xlabel("Durée de séjour (jours)")
plt.ylabel("Nombre de patients")
plt.axvline(df["time_in_hospital"].mean(), color="red", linestyle="--",
            label=f"Moyenne = {df['time_in_hospital'].mean():.1f} j")
plt.axvline(SEUIL_JOURS + 0.5, color="green", linestyle="-",
            label=f"Seuil court / long : {SEUIL_JOURS} j")
plt.legend()
plt.show()

print(df["time_in_hospital"].describe())

In [ ]:
numeric_cols = [
    "num_lab_procedures", "num_procedures", "num_medications",
    "number_outpatient", "number_emergency", "number_inpatient",
    "number_diagnoses",
]

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.ravel()
for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], bins=30, ax=axes[i], color="teal")
    axes[i].set_title(f"Histogramme — {col}")
fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.ravel()
for i, col in enumerate(numeric_cols):
    sns.boxplot(x=df[col], ax=axes[i], color="lightsalmon")
    axes[i].set_title(f"Boxplot — {col}")
fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()


**Lecture rapide :** `number_outpatient`, `number_emergency` et `number_inpatient` sont très
asymétriques (beaucoup de 0, quelques valeurs extrêmes) → nécessitent un plafonnement et une
transformation logarithmique. `num_lab_procedures` et `num_medications` sont plus proches d'une
distribution régulière mais comportent aussi des valeurs extrêmes à plafonner.

### 1.3 Construction de la cible binaire `long_stay`

On transforme la durée (nombre de jours) en deux classes :

| Classe | Valeur de `long_stay` | Définition |
|---|---|---|
| Courte durée | 0 | `time_in_hospital` ≤ `SEUIL_JOURS` |
| Longue durée | 1 | `time_in_hospital` > `SEUIL_JOURS` |

Le seuil de 3 jours correspond à la médiane : il donne deux classes équilibrées, ce qui rend
l'accuracy fiable (avec un seuil très élevé, par exemple 7 jours, un modèle qui prédirait
toujours « court » aurait déjà une accuracy proche de 86 %).

In [ ]:
df["long_stay"] = (df["time_in_hospital"] > SEUIL_JOURS).astype(int)
LABELS = {0: "Courte durée", 1: "Longue durée"}

repartition = df["long_stay"].map(LABELS).value_counts()
print(repartition)
print((df["long_stay"].value_counts(normalize=True).sort_index() * 100).round(1).rename(LABELS))

plt.figure(figsize=(5, 4))
sns.barplot(x=repartition.index, y=repartition.values, palette=["#4C78A8", "#E45756"],
            hue=repartition.index, legend=False)
plt.title(f"Répartition des classes (seuil = {SEUIL_JOURS} jours)")
plt.ylabel("Nombre de patients")
plt.show()

## 2. Transformation

### 2.0 Variables à exclure et colonnes hors scope
- `readmitted` est une information future (connue seulement après la sortie) → exclue de tout
  le jeu de variables explicatives, quelle que soit la version.
- `time_in_hospital` est retirée des variables explicatives, car la cible `long_stay` en est
  dérivée (la garder donnerait au modèle la réponse).

In [ ]:
df_model = df.drop(columns=["readmitted"]).copy()
print(df_model.shape)


### 2.1 Découpage train / test (80/20)
Fait **avant** tout calcul ajusté (percentiles, scaler) pour éviter toute fuite d'information
du test vers l'entraînement. Le découpage est **stratifié** sur `long_stay` : les deux classes
ont la même proportion dans le train et dans le test.

In [ ]:
y = df_model["long_stay"]
X = df_model.drop(columns=["time_in_hospital", "long_stay"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train :", X_train.shape, " | Test :", X_test.shape)
print("Part de « longue durée » — train :", round(y_train.mean() * 100, 1), "% | test :",
      round(y_test.mean() * 100, 1), "%")

### 2.2 Valeurs extrêmes — plafonnement au 99e percentile
Calculé **sur l'entraînement uniquement**, puis appliqué à `train` et `test` (plutôt que
suppression des lignes).

In [ ]:
count_vars = [
    "num_lab_procedures", "num_procedures", "num_medications",
    "number_outpatient", "number_emergency", "number_inpatient",
    "number_diagnoses",
]

p99 = X_train[count_vars].quantile(0.99)
print("Seuils du 99e percentile (train) :")
print(p99)

for col in count_vars:
    X_train[col] = X_train[col].clip(upper=p99[col])
    X_test[col] = X_test[col].clip(upper=p99[col])


### 2.3 Asymétrie — transformation log(1+x)
Appliquée à `number_outpatient`, `number_emergency`, `number_inpatient`, `num_medications`
(après plafonnement).

In [ ]:
log_vars = ["number_outpatient", "number_emergency", "number_inpatient", "num_medications"]

for col in log_vars:
    X_train[f"log_{col}"] = np.log1p(X_train[col])
    X_test[f"log_{col}"] = np.log1p(X_test[col])

X_train[[f"log_{c}" for c in log_vars]].describe()


### 2.4 Encodage

- **Variables binaires → 0/1** : `gender`, `change`, `diabetesMed`
- **One-hot** : `medical_specialty` et les diagnostics (`diag_1`, `diag_2`, `diag_3`)

Les codes diagnostics bruts (ICD-9) sont très nombreux (plusieurs centaines de valeurs
distinctes) : un one-hot direct créerait des centaines de colonnes quasi vides. On les regroupe
d'abord en grandes catégories cliniques standard (Circulatoire, Respiratoire, Digestif,
Diabète, Blessure, Musculo-squelettique, Génito-urinaire, Néoplasme, Autre), puis on applique
le one-hot sur ces catégories — c'est l'approche usuelle pour ce jeu de données.

In [ ]:
def map_icd9_category(code):
    """Regroupe un code ICD-9 (diag_1/2/3) en grande catégorie clinique."""
    if pd.isna(code):
        return "Missing"
    code = str(code)
    if code.startswith("V") or code.startswith("E"):
        return "Other"
    try:
        val = float(code)
    except ValueError:
        return "Other"

    if val == 250 or (250 <= val < 251):
        return "Diabetes"
    if (390 <= val <= 459) or val == 785:
        return "Circulatory"
    if (460 <= val <= 519) or val == 786:
        return "Respiratory"
    if (520 <= val <= 579) or val == 787:
        return "Digestive"
    if (800 <= val <= 999):
        return "Injury"
    if (710 <= val <= 739):
        return "Musculoskeletal"
    if (580 <= val <= 629) or val == 788:
        return "Genitourinary"
    if (140 <= val <= 239):
        return "Neoplasms"
    return "Other"


for col in ["diag_1", "diag_2", "diag_3"]:
    X_train[f"{col}_cat"] = X_train[col].apply(map_icd9_category)
    X_test[f"{col}_cat"] = X_test[col].apply(map_icd9_category)

X_train["diag_1_cat"].value_counts()


In [ ]:
# medical_specialty : on regroupe les catégories rares (hors top 10 + "Unknown") dans "Other"
top_specialties = (
    X_train.loc[X_train["medical_specialty"] != "Unknown", "medical_specialty"]
    .value_counts()
    .head(10)
    .index
)

def map_specialty(spec):
    if pd.isna(spec) or spec == "Unknown":
        return "Unknown"
    return spec if spec in top_specialties else "Other"

X_train["medical_specialty_grp"] = X_train["medical_specialty"].apply(map_specialty)
X_test["medical_specialty_grp"] = X_test["medical_specialty"].apply(map_specialty)

X_train["medical_specialty_grp"].value_counts()


In [ ]:
# Variables binaires -> 0/1
binary_map_gender = {"Female": 0, "Male": 1}
binary_map_change = {"No": 0, "Ch": 1}
binary_map_diabmed = {"No": 0, "Yes": 1}

for Xset in (X_train, X_test):
    Xset["gender_bin"] = Xset["gender"].map(binary_map_gender)
    Xset["change_bin"] = Xset["change"].map(binary_map_change)
    Xset["diabetesMed_bin"] = Xset["diabetesMed"].map(binary_map_diabmed)


In [ ]:
# One-hot pour diagnostics (catégories) et spécialité (catégorie regroupée)
cat_cols = ["diag_1_cat", "diag_2_cat", "diag_3_cat", "medical_specialty_grp"]

X_train_oh = pd.get_dummies(X_train[cat_cols], prefix=cat_cols)
X_test_oh = pd.get_dummies(X_test[cat_cols], prefix=cat_cols)
# aligner les colonnes du test sur celles du train (catégories absentes du test -> 0)
X_test_oh = X_test_oh.reindex(columns=X_train_oh.columns, fill_value=0)

print(X_train_oh.shape, X_test_oh.shape)
X_train_oh.head()


### 2.5 Normalisation — StandardScaler
Ajusté sur le jeu d'entraînement uniquement, puis appliqué à `train` et `test`.

In [ ]:
scale_cols = [
    "num_lab_procedures", "num_procedures", "number_diagnoses",
    "log_number_outpatient", "log_number_emergency",
    "log_number_inpatient", "log_num_medications",
]

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train[scale_cols]),
    columns=[f"{c}_scaled" for c in scale_cols],
    index=X_train.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test[scale_cols]),
    columns=[f"{c}_scaled" for c in scale_cols],
    index=X_test.index,
)

X_train_scaled.describe()


In [ ]:
# Assemblage du dataframe final de features (numériques scalées + binaires + one-hot)
feature_binary = ["gender_bin", "change_bin", "diabetesMed_bin"]

X_train_full = pd.concat(
    [X_train_scaled, X_train[feature_binary], X_train_oh], axis=1
)
X_test_full = pd.concat(
    [X_test_scaled, X_test[feature_binary], X_test_oh], axis=1
)

print("Nombre total de features après préparation :", X_train_full.shape[1])
X_train_full.head()


## 3. Sélection de variables

### 3.1 Matrice de corrélation avec `long_stay`
On conserve les variables dont la corrélation (en valeur absolue) avec la cible binaire
`long_stay` dépasse **0,1** (corrélation de Pearson avec une cible 0/1, dite point-bisériale).
Si deux variables conservées sont corrélées entre elles à plus de **0,8**, une seule est
gardée.

In [ ]:
corr_with_target = (
    X_train_full.assign(long_stay=y_train)
    .corr(numeric_only=True)["long_stay"]
    .drop("long_stay")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)
corr_with_target

In [ ]:
plt.figure(figsize=(8, 10))
corr_with_target.plot(kind="barh")
plt.axvline(0.1, color="red", linestyle="--", linewidth=1)
plt.axvline(-0.1, color="red", linestyle="--", linewidth=1)
plt.title("Corrélation de chaque variable avec long_stay")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
CORR_THRESHOLD = 0.1
REDUNDANCY_THRESHOLD = 0.8

candidates = corr_with_target[corr_with_target.abs() > CORR_THRESHOLD].index.tolist()
print(f"{len(candidates)} variable(s) au-dessus du seuil de corrélation ({CORR_THRESHOLD}) :")
print(candidates)


In [ ]:
# Suppression des redondances : si deux variables candidates sont corrélées entre elles
# à plus de 0.8, on ne garde que celle la plus corrélée avec la cible.
pairwise_corr = X_train_full[candidates].corr().abs()

to_drop = set()
for i, var_i in enumerate(candidates):
    for var_j in candidates[i + 1:]:
        if var_i in to_drop or var_j in to_drop:
            continue
        if pairwise_corr.loc[var_i, var_j] > REDUNDANCY_THRESHOLD:
            # on garde celle qui a la plus forte corrélation avec la cible
            weaker = var_i if abs(corr_with_target[var_i]) < abs(corr_with_target[var_j]) else var_j
            to_drop.add(weaker)
            print(f"Redondance {var_i} / {var_j} (corr={pairwise_corr.loc[var_i, var_j]:.2f}) "
                  f"-> suppression de {weaker}")

selected_features = [c for c in candidates if c not in to_drop]
print(f"\n{len(selected_features)} variable(s) retenue(s) après filtrage des redondances :")
print(selected_features)


### 3.2 Deux versions du jeu de variables

- **« Admission »** : variables disponibles dès l'admission — sans `num_lab_procedures` ni
  `num_medications` (ni leurs dérivées).
- **« Après séjour »** : toutes les variables sélectionnées, y compris celles connues
  seulement en fin de séjour.

In [ ]:
admission_exclude_prefixes = ("num_lab_procedures", "log_num_medications", "num_medications")

features_admission = [
    f for f in selected_features
    if not any(f.startswith(p) for p in admission_exclude_prefixes)
]
features_after_stay = selected_features  # toutes les variables retenues

print(f"Version « Admission »   : {len(features_admission)} variable(s)")
print(features_admission)
print()
print(f"Version « Après séjour » : {len(features_after_stay)} variable(s)")
print(features_after_stay)


In [ ]:
X_train_admission = X_train_full[features_admission]
X_test_admission = X_test_full[features_admission]

X_train_after_stay = X_train_full[features_after_stay]
X_test_after_stay = X_test_full[features_after_stay]

print("Admission   — train/test :", X_train_admission.shape, X_test_admission.shape)
print("Après séjour — train/test :", X_train_after_stay.shape, X_test_after_stay.shape)


## 4. Export — jeux de données prêts pour la modélisation

Sauvegarde des quatre jeux de données (deux versions × train/test) ainsi que la cible binaire
`long_stay`, prêts pour l'étape de modélisation (XGBoost et AdaBoost).

In [ ]:
X_train_admission.to_csv("X_train_admission.csv", index=False)
X_test_admission.to_csv("X_test_admission.csv", index=False)
X_train_after_stay.to_csv("X_train_after_stay.csv", index=False)
X_test_after_stay.to_csv("X_test_after_stay.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Fichiers exportés :")
for f in ["X_train_admission.csv", "X_test_admission.csv",
          "X_train_after_stay.csv", "X_test_after_stay.csv",
          "y_train.csv", "y_test.csv"]:
    print(" -", f)


Les deux jeux de variables (`features_admission` et `features_after_stay` ci-dessus)
sont prêts pour l'entraînement : XGBoost et AdaBoost, évalués via l'accuracy, la précision,
le rappel, le F1, l'AUC et la matrice de confusion.

## 5. Modélisation

Deux modèles de classification sont entraînés sur chacune des deux versions du jeu de
variables (« Admission » et « Après séjour »), plus une **référence** qui prédit toujours la
classe majoritaire (elle sert de point de comparaison pour l'accuracy) :

- **AdaBoost** (`AdaBoostClassifier`, 200 arbres de profondeur 1 par défaut) — chaque nouvel arbre
  donne plus de poids aux patients mal classés par les précédents
- **XGBoost** (`XGBClassifier`) — arbres successifs qui corrigent les erreurs des précédents, avec
  régularisation ; en général plus performant, mais aussi plus sensible au surapprentissage
- **Référence (classe majoritaire)** (`DummyClassifier`) — ce que l'on obtiendrait sans modèle

In [ ]:
REF = "Référence (classe majoritaire)"

def build_models():
    return {
        REF: DummyClassifier(strategy="most_frequent"),
        "AdaBoost": AdaBoostClassifier(
            n_estimators=200, learning_rate=0.5, random_state=RANDOM_STATE
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1,
        ),
    }

datasets = {
    "Admission": (X_train_admission, X_test_admission),
    "Après séjour": (X_train_after_stay, X_test_after_stay),
}

results = []        # métriques de test, une ligne par (version, modèle)
fitted_models = {}   # pour réutilisation (ex. importance des variables)
predictions = {}     # classes et probabilités prédites sur le test

for version, (X_tr, X_te) in datasets.items():
    for model_name, model in build_models().items():
        model.fit(X_tr, y_train)
        pred_train = model.predict(X_tr)
        pred_test = model.predict(X_te)
        proba_test = model.predict_proba(X_te)[:, 1]

        fitted_models[(version, model_name)] = model
        predictions[(version, model_name)] = {"pred": pred_test, "proba": proba_test}

        results.append({
            "Version": version,
            "Modèle": model_name,
            "Accuracy_train": accuracy_score(y_train, pred_train),
            "Accuracy_test": accuracy_score(y_test, pred_test),
            "Precision_test": precision_score(y_test, pred_test, zero_division=0),
            "Rappel_test": recall_score(y_test, pred_test, zero_division=0),
            "F1_test": f1_score(y_test, pred_test, zero_division=0),
            "AUC_test": roc_auc_score(y_test, proba_test),
        })

print(f"{len(results)} combinaisons (version × modèle) entraînées.")

## 6. Évaluation

### 6.1 Tableau comparatif
**Accuracy** sur le test (part de patients bien classés, court ou long), puis précision, rappel
et F1 de la classe « longue durée », et AUC. La ligne « Référence » montre l'accuracy que l'on
obtient sans modèle : un modèle n'est utile que s'il fait nettement mieux.

In [ ]:
results_df = pd.DataFrame(results)
results_df["Surapprentissage"] = results_df["Accuracy_train"] - results_df["Accuracy_test"]

display_cols = ["Version", "Modèle", "Accuracy_test", "Precision_test", "Rappel_test",
                "F1_test", "AUC_test", "Surapprentissage"]
(results_df[display_cols]
 .sort_values(["Version", "Accuracy_test"], ascending=[True, False])
 .round(3))

**Lecture :**
- `Accuracy_test` : part de patients correctement classés en courte / longue durée. À comparer à
  la ligne « Référence » (classe majoritaire) : l'écart mesure l'apport réel du modèle.
- `Rappel_test` : part des vrais séjours longs que le modèle détecte. C'est la métrique la plus
  importante pour l'objectif métier (anticiper les lits occupés longtemps).
- `Precision_test` : parmi les séjours annoncés longs, part qui le sont vraiment (fausses alertes).
- `AUC_test` : capacité à classer les patients du moins au plus susceptible de rester longtemps
  (0,5 = hasard, 1 = parfait).
- `Surapprentissage` (= Accuracy_train − Accuracy_test) : une valeur nettement positive signale
  un modèle qui a trop appris le jeu d'entraînement.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, version in zip(axes, datasets.keys()):
    subset = results_df[results_df["Version"] == version].sort_values("Accuracy_test")
    sns.barplot(data=subset, x="Modèle", y="Accuracy_test", ax=ax, color="steelblue")
    ax.bar_label(ax.containers[0], fmt="%.3f")
    ref_acc = subset.loc[subset["Modèle"] == REF, "Accuracy_test"].iloc[0]
    ax.axhline(ref_acc, color="red", linestyle="--", linewidth=1,
               label=f"Référence : {ref_acc:.3f}")
    ax.set_title(f"Accuracy sur le test — {version}")
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0.4, 1)
    ax.tick_params(axis="x", rotation=15)
    ax.legend()
plt.tight_layout()
plt.show()

### 6.2 Meilleur modèle par version

In [ ]:
best_per_version = (
    results_df[results_df["Modèle"] != REF]
    .sort_values("Accuracy_test", ascending=False)
    .groupby("Version")
    .first()[["Modèle", "Accuracy_test", "Rappel_test", "F1_test", "AUC_test"]]
)
best_per_version

### 6.3 Matrice de confusion — meilleur modèle de chaque version
La matrice montre où le modèle se trompe : les séjours longs annoncés courts (en bas à gauche)
sont les erreurs les plus coûteuses pour la planification des lits.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, version in zip(axes, datasets.keys()):
    name = best_per_version.loc[version, "Modèle"]
    ConfusionMatrixDisplay.from_predictions(
        y_test, predictions[(version, name)]["pred"],
        display_labels=["Courte durée", "Longue durée"], cmap="Blues", ax=ax, colorbar=False,
    )
    ax.set_title(f"{name} — {version}")
    ax.grid(False)
plt.tight_layout()
plt.show()

for version in datasets.keys():
    name = best_per_version.loc[version, "Modèle"]
    print(f"===== {name} — {version} =====")
    print(classification_report(y_test, predictions[(version, name)]["pred"],
                                target_names=["Courte durée", "Longue durée"], digits=3))

### 6.4 Importance des variables (XGBoost, version « Après séjour »)
Quelles variables pèsent le plus dans la décision « court » ou « long » ?

In [ ]:
xgb_model = fitted_models[("Après séjour", "XGBoost")]
importances = pd.Series(xgb_model.feature_importances_, index=X_train_after_stay.columns)

plt.figure(figsize=(8, 5))
importances.sort_values().tail(15).plot(kind="barh", color="steelblue")
plt.title("Importance des variables — XGBoost (Après séjour)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

### 6.5 Export des résultats

In [ ]:
results_df.to_csv("resultats_modeles.csv", index=False)
print("Fichier exporté : resultats_modeles.csv")


---
### Résumé

| Version | # variables | Variables exclues |
|---|---|---|
| Admission | voir `features_admission` ci-dessus | `num_lab_procedures`, `num_medications` (et dérivées) |
| Après séjour | voir `features_after_stay` ci-dessus | — |

**Problème :** classification binaire, `long_stay` = 1 si `time_in_hospital` > `SEUIL_JOURS`
(3 jours par défaut). **Modèles :** XGBoost et AdaBoost, comparés à une référence qui prédit
toujours la classe majoritaire, sur les deux versions du jeu de variables.

Le tableau `results_df`, le graphique d'accuracy et les matrices de confusion ci-dessus donnent
le détail ; `best_per_version` résume le meilleur modèle de chaque version. Les valeurs exactes
dépendent des variables réellement sélectionnées lors de l'exécution (seuil de corrélation 0,1).